# Automatic chord recognition

This notebook presents the hierarchical chord-recognition system, reproduces its RWC evaluation, and provides an in-notebook audio player with synchronized chord predictions.

The default system is a hierarchical CQT72 CNN followed by a second-order HSMM. The CNN predicts root, quality, seventh, and extension separately; the HSMM uses the current emissions and the two previous chords to select a musically plausible sequence.

## 1. Setup (Jupyter, VS Code, or Google Colab)

Open this notebook in Jupyter Notebook/JupyterLab, VS Code with Jupyter support,
or Google Colab. Use a **Python 3.11 or newer** kernel and choose **Run All**
(**Runtime → Run all** in Colab).

The same setup runs in all three environments. It downloads the project from a
fixed public GitHub revision (about 170 MB), verifies the download, and installs
application packages into the selected Python kernel. Internet access and
permission to install packages are required. No other project files, uploads,
Google Drive mount, or project-specific virtual environment are needed.
CPU works; an available CUDA GPU can accelerate CNN inference.

The project and generated results are stored in the system temporary directory;
the setup prints their location. Temporary files may be deleted by the host,
so save any results you want to keep. Run setup again in a fresh environment.
If installation requests a kernel/runtime restart, restart it and run all cells
again. Installing dependencies may update packages in the selected environment.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile

REVISION = "cbbde3575a52a9183feab04ef06df567320f1a20"
ARCHIVE_URL = f"https://codeload.github.com/hanneswesterberg/dt2470-project/zip/{REVISION}"
ARCHIVE_SHA256 = "1370a617403fd9d716fb246cd56ebb31bd36faeb4b5c1ba6fd73aebecf0abd6f"
ROOT = Path(tempfile.gettempdir()) / "dt2470_submission"
marker = ROOT / ".source-revision"

if not marker.is_file() or marker.read_text() != REVISION:
    print("Downloading the project, models, evaluation features, and example audio …")
    with tempfile.TemporaryDirectory() as directory:
        archive = Path(directory) / "project.zip"
        urllib.request.urlretrieve(ARCHIVE_URL, archive)
        with archive.open("rb") as handle:
            digest = hashlib.file_digest(handle, "sha256").hexdigest()
        if digest != ARCHIVE_SHA256:
            raise RuntimeError("Project download checksum mismatch. Run setup again.")
        prefix = f"dt2470-project-{REVISION}/submission/"
        included = {"src", "data", "models", "audio", "features", "annotations",
                    "main.py", "requirements.txt", "ARCHITECTURE.md"}
        with zipfile.ZipFile(archive) as bundle:
            for member in bundle.infolist():
                if member.is_dir() or not member.filename.startswith(prefix):
                    continue
                relative = Path(member.filename[len(prefix):])
                target = (ROOT / relative).resolve()
                if not target.is_relative_to(ROOT.resolve()):
                    raise ValueError(f"Unsafe archive path: {member.filename}")
                if relative.parts[0] not in included and relative.parts[0] != "training":
                    continue
                # Only the evaluator is needed; training requires the full training dataset.
                if relative.parts[0] == "training" and relative.name != "evaluate_rwc.py":
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with bundle.open(member) as source, target.open("wb") as destination:
                    shutil.copyfileobj(source, destination)
    marker.write_text(REVISION)

# The running notebook already has a frontend and kernel; install application packages only.
requirements = [line for line in (ROOT / "requirements.txt").read_text().splitlines()
                if line.strip() and not line.startswith(("#", "ipykernel", "jupyterlab"))]
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                *requirements], check=True)
print("Project ready:", ROOT)

In [ ]:
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT)
env["MPLBACKEND"] = "Agg"
print("CNN device:", "cuda" if torch.cuda.is_available() else "cpu")

## 2. Architecture and musical motivation

A chord is not only a single class. Its root describes the harmonic centre, quality describes its basic colour, the seventh changes its function and tension, and extensions add further colour. Predicting these parts with separate but connected heads gives the model useful structure and makes errors easier to interpret.

The CQT keeps pitch relationships visible. The CNN turns short time-frequency windows into probabilities. The second-order HSMM smooths those probabilities over time while modelling chord transitions and durations. This avoids isolated one-frame chord changes and captures common progressions such as tonic–predominant–dominant movement. See the [architecture description](https://github.com/hanneswesterberg/dt2470-project/blob/cbbde3575a52a9183feab04ef06df567320f1a20/submission/ARCHITECTURE.md) for details.

## 3. Training and the supplied checkpoint

This presentation uses the supplied trained hierarchical CNN, so no training is
required. The downloaded cached features contain the fixed 15-song evaluation
split. Re-training requires the full training feature dataset, which is not
included in this presentation. The [original training entry point](https://github.com/hanneswesterberg/dt2470-project/blob/cbbde3575a52a9183feab04ef06df567320f1a20/submission/training/train_hierarchical_cnn.py)
documents the training procedure.

## 4. RWC evaluation

The evaluation uses the fixed 15-song test split in `training/evaluate_rwc.py`. **Frame accuracy** gives every analysis frame equal weight. **Duration-weighted accuracy** gives each frame its actual time span, so it represents the fraction of annotated listening time recognized correctly. Results are shown at three levels: root, root + quality, and the complete root + quality + seventh label.

In [ ]:
FEATURE_DIR = ROOT / 'features'
assert FEATURE_DIR.is_dir(), f'Feature directory is missing: {FEATURE_DIR}'
subprocess.run(
    [sys.executable, 'training/evaluate_rwc.py',
     '--feature-dir', str(FEATURE_DIR),
     '--checkpoint', 'models/best_model.pt'],
    cwd=ROOT, env=env, check=True,
)
report = json.loads((ROOT / 'output/rwc_test_evaluation.json').read_text())
print(f"Evaluated {len(report['per_song'])} songs.")

### Aggregate metrics

The table and chart contain every metric written by the evaluator. Values are macro averages: every test song contributes equally.

In [ ]:
metric_info = [
    ('root_accuracy', 'Root', 'Frame accuracy'),
    ('root_quality_accuracy', 'Root + quality', 'Frame accuracy'),
    ('root_quality_seventh_accuracy', 'Root + quality + seventh', 'Frame accuracy'),
    ('song_root_percent', 'Root', 'Duration-weighted accuracy'),
    ('song_root_quality_percent', 'Root + quality', 'Duration-weighted accuracy'),
    ('song_root_quality_seventh_percent', 'Root + quality + seventh', 'Duration-weighted accuracy'),
]
metrics = report['cnn_macro_average']
aggregate = pd.DataFrame([
    {'Evaluation': evaluation, 'Chord components': components, 'Score': metrics[key]}
    for key, components, evaluation in metric_info
])
display(aggregate.style.hide(axis='index').format({'Score': '{:.2%}'}).bar(subset=['Score'], vmin=0, vmax=1, color='#66c7a5'))

fig, ax = plt.subplots(figsize=(10, 4.8))
plot_labels = [f'{row.Evaluation} — {row["Chord components"]}' for _, row in aggregate.iterrows()]
values = 100 * aggregate['Score'].to_numpy()
colors = ['#5fa8d3'] * 3 + ['#e9b949'] * 3
bars = ax.barh(plot_labels[::-1], values[::-1], color=colors[::-1])
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontweight='bold')
ax.set(xlim=(0, 100), xlabel='Accuracy (%)', title='RWC test-set macro averages')
ax.grid(axis='x', alpha=.25)
ax.spines[['top', 'right', 'left']].set_visible(False)
plt.tight_layout()
plt.show()

### Per-song metrics

This view exposes all six metrics for every test song, making the spread behind the macro average visible.

In [ ]:
short_names = ['Frame: R', 'Frame: R+Q', 'Frame: R+Q+7', 'Duration: R', 'Duration: R+Q', 'Duration: R+Q+7']
per_song = pd.DataFrame(
    [[f"RWC P{item['song']:03d}"] + [item['cnn'][key] for key, _, _ in metric_info]
     for item in report['per_song']],
    columns=['Song'] + short_names,
).set_index('Song')
display(per_song.style.format('{:.2%}').background_gradient(cmap='YlGn', vmin=0, vmax=1))

fig, ax = plt.subplots(figsize=(11, 6.5))
image = ax.imshow(100 * per_song.to_numpy(), cmap='YlGn', vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(len(short_names)), short_names, rotation=25, ha='right')
ax.set_yticks(range(len(per_song)), per_song.index)
for row in range(len(per_song)):
    for column in range(len(short_names)):
        value = 100 * per_song.iloc[row, column]
        ax.text(column, row, f'{value:.0f}', ha='center', va='center', fontsize=8, color='white' if value < 55 else '#132a13')
fig.colorbar(image, ax=ax, label='Accuracy (%)', pad=.02)
ax.set_title('All evaluation metrics by test song')
plt.tight_layout()
plt.show()

## 5. Listen to and visualize a song

The default is an annotated RWC song. Inference writes the chord sequence, then the next cells show the waveform, the model prediction, and ground truth on the same time axis. The embedded player keeps the audio, live predicted chord, live ground-truth chord, and both colored timelines synchronized; click any chord block to seek.

In [ ]:
supported_audio = {'.wav', '.mp3', '.flac', '.ogg'}
audio_files = [path for path in sorted((ROOT / 'audio').glob('*'))
               if path.suffix.lower() in supported_audio and path.stat().st_size > 0]
display(pd.DataFrame({
    'Available song': [path.stem for path in audio_files],
    'Format': [path.suffix.lower().lstrip('.').upper() for path in audio_files],
    'Size (MB)': [path.stat().st_size / 1_000_000 for path in audio_files],
}).style.hide(axis='index').format({'Size (MB)': '{:.1f}'}))

In [ ]:
# Choose any listed song. Annotated RWC examples show ground truth; other songs run prediction-only.
song = 'RWC_P017'
assert any(song in {path.name, path.stem} for path in audio_files), f'Audio not found: {song}'
subprocess.run(
    [sys.executable, 'main.py', '--source', 'local', '--input-dir', 'audio',
     '--cnn-checkpoint', 'models/best_model.pt',
     '--language-model', 'models/harmonic_model.pkl',
     '--split-manifest', 'models/split_manifest.json',
     '--song', song, '--non-interactive'],
    cwd=ROOT, env=env, check=True,
)
summary = json.loads((ROOT / 'output/summary.json').read_text())
result = next(item for item in summary if item['song'] == Path(song).stem)
audio_path = Path(result['audio'])
if not audio_path.is_absolute():
    audio_path = ROOT / audio_path
timeline_path = Path(result['prediction_csv'])
if not timeline_path.is_absolute():
    timeline_path = ROOT / timeline_path
timeline = pd.read_csv(timeline_path)
annotation_path = ROOT / 'annotations' / f'{audio_path.stem}.csv'
ground_truth = None
if annotation_path.is_file():
    ground_truth = pd.read_csv(annotation_path, sep=';').rename(
        columns={'t_start': 'start_seconds', 't_end': 'end_seconds'},
    )
    print(f"Recognized {len(timeline)} chord events and loaded {len(ground_truth)} ground-truth events for {audio_path.name}.")
else:
    print(f"Recognized {len(timeline)} chord events for {audio_path.name}; no ground truth is available.")

In [ ]:
# A lightweight waveform is sufficient for display and keeps this cell responsive.
waveform, sample_rate = librosa.load(audio_path, sr=8_000, mono=True)
max_points = 60_000
stride = max(1, len(waveform) // max_points)
times = np.arange(0, len(waveform), stride) / sample_rate
duration = len(waveform) / sample_rate

roots = ['C#', 'Eb', 'F#', 'Ab', 'Bb', 'C', 'D', 'E', 'F', 'G', 'A', 'B', 'N']
palette = plt.get_cmap('tab20')(np.linspace(0, 1, len(roots)))
def chord_color(label):
    root = next((root for root in roots[:-1] if str(label).startswith(root)), 'N')
    return palette[roots.index(root)]

if ground_truth is not None:
    fig, (wave_ax, predicted_ax, truth_ax) = plt.subplots(3, 1, figsize=(15, 7), sharex=True,
        gridspec_kw={'height_ratios': [3, 1, 1], 'hspace': .08})
    plot_axes = (wave_ax, predicted_ax, truth_ax)
    title = f'Audio, model prediction, and ground truth — {audio_path.stem}'
else:
    fig, (wave_ax, predicted_ax) = plt.subplots(2, 1, figsize=(15, 5.5), sharex=True,
        gridspec_kw={'height_ratios': [3, 1], 'hspace': .08})
    truth_ax = None
    plot_axes = (wave_ax, predicted_ax)
    title = f'Audio and model prediction — {audio_path.stem}'
wave_ax.plot(times, waveform[::stride], color='#263238', linewidth=.45)
wave_ax.set(ylabel='Amplitude', title=title)
wave_ax.grid(alpha=.16)

label_threshold = max(2.5, duration / 55)
for event in timeline.itertuples(index=False):
    start, end, chord = event.start_seconds, event.end_seconds, str(event.chord)
    color = chord_color(chord)
    wave_ax.axvspan(start, end, color=color, alpha=.12, linewidth=0)
    predicted_ax.axvspan(start, end, color=color, alpha=.9, ec='white', lw=.5)
    if end - start >= label_threshold:
        predicted_ax.text((start + end) / 2, .5, chord, ha='center', va='center',
                      fontsize=8, fontweight='bold', rotation=90 if end - start < duration / 28 else 0)

if ground_truth is not None:
    for event in ground_truth.itertuples(index=False):
        start, end, chord = event.start_seconds, event.end_seconds, str(event.chord)
        display_chord = chord.replace(':maj', '').replace(':min', 'min')
        truth_ax.axvspan(start, end, color=chord_color(display_chord), alpha=.9, ec='white', lw=.5)
        if end - start >= label_threshold:
            truth_ax.text((start + end) / 2, .5, chord, ha='center', va='center',
                          fontsize=8, fontweight='bold', rotation=90 if end - start < duration / 28 else 0)

predicted_ax.set(xlim=(0, duration), ylim=(0, 1), yticks=[])
predicted_ax.set_ylabel('Predicted', rotation=0, ha='right', va='center')
if truth_ax is not None:
    truth_ax.set(xlim=(0, duration), ylim=(0, 1), yticks=[], xlabel='Time (seconds)')
    truth_ax.set_ylabel('Ground truth', rotation=0, ha='right', va='center')
else:
    predicted_ax.set_xlabel('Time (seconds)')
for axis in plot_axes:
    axis.spines[['top', 'right']].set_visible(False)
plt.show()

### Interactive audio and chord player

The player appears directly below. Press Play to hear the song; click a chord
block to seek. Predictions and available ground truth stay synchronized.
Audio is embedded in the player so it does not depend on browser access to
files on the Python runtime.

Allow trusted HTML/JavaScript output to use the interactive player. In Jupyter
or VS Code, trust the notebook/workspace when prompted.

In [ ]:
import base64
import html
import io
import re
import soundfile as sf
from urllib.parse import unquote

viewer_path = ROOT / "output" / f"{Path(song).stem}_viewer.html"
assert viewer_path.is_file(), f"Viewer was not generated: {viewer_path}"
viewer_html = viewer_path.read_text(encoding="utf-8")
audio_tag = re.search(r'<audio\b[^>]*\bsrc="([^"]+)"', viewer_html)
assert audio_tag is not None, "Generated viewer has no audio source."
media_path = (viewer_path.parent / unquote(html.unescape(audio_tag.group(1)))).resolve()
assert media_path.is_relative_to(ROOT.resolve()), "Audio source is outside the project."
# Compress only playback audio to keep the notebook output small.
playback, playback_rate = librosa.load(media_path, sr=22_050, mono=True)
audio_buffer = io.BytesIO()
# Small writes avoid encoder crashes on long songs in some libsndfile versions.
with sf.SoundFile(audio_buffer, mode="w", samplerate=playback_rate, channels=1,
                  format="OGG", subtype="VORBIS") as compressed:
    for start in range(0, len(playback), 65_536):
        compressed.write(playback[start:start + 65_536])
audio_data = base64.b64encode(audio_buffer.getvalue()).decode("ascii")
viewer_html = viewer_html.replace(
    f'src="{audio_tag.group(1)}"', f'src="data:audio/ogg;base64,{audio_data}"', 1,
)
# srcdoc keeps the existing player scripts and styles inside their own document.
display(HTML(
    '<iframe title="Synchronized audio and chord player" '
    'sandbox="allow-scripts allow-same-origin" '
    'style="width:100%;height:1100px;border:0" '
    f'srcdoc="{html.escape(viewer_html, quote=True)}"></iframe>'
))